In [1]:
"""
f = min(x1,x3) + max(x1, x3, x2+x4)  ->  QUBO  ->  Ising (Pauli I/Z strings)

pip install qiskit qiskit-optimization numpy
"""
import numpy as np
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo

U = 2                 # assumed upper bound on every x_i (x_i in 1..U). Must come from your problem.
M = 2 * U + 1         # penalty weight: must exceed the most the objective can gain by cheating

# ---------------------------------------------------------------- 1. model
qp = QuadraticProgram("minmax")

for i in range(1, 5):
    qp.integer_var(lowerbound=1, upperbound=U, name=f"x{i}")
qp.integer_var(lowerbound=1, upperbound=2 * U, name="t")   # t ~ max(x1, x3, x2+x4)
qp.binary_var(name="b")                                    # b selects min(x1, x3)

# objective: [x3 + b*(x1 - x3)]  (= min(x1,x3))  +  t  (= max(...))
# TODO: add the rest of YOUR objective here (it must enter with a positive coefficient).
qp.minimize(
    linear={"x3": 1, "t": 1},
    quadratic={("b", "x1"): 1, ("b", "x3"): -1},
)

# t >= each argument of the max (the "+ t" in the objective pushes t down onto the max)
qp.linear_constraint({"t": 1, "x1": -1},           ">=", 0, "t_ge_x1")
qp.linear_constraint({"t": 1, "x3": -1},           ">=", 0, "t_ge_x3")
qp.linear_constraint({"t": 1, "x2": -1, "x4": -1}, ">=", 0, "t_ge_x2_plus_x4")

# TODO: add your other constraints here, e.g.
# qp.linear_constraint({"x1": 1, "x2": 1, "x3": 1, "x4": 1}, "==", N, "total")

# ---------------------------------------------------------------- 2. QUBO
# Converter does: integer -> binary expansion, inequality -> slack bits,
# constraints -> M * (...)^2 penalties.
conv = QuadraticProgramToQubo(penalty=M)
qubo = conv.convert(qp)
print("binary variables / qubits:", qubo.get_num_vars())

# ---------------------------------------------------------------- 3. Ising
op, offset = qubo.to_ising()      # op: SparsePauliOp with only I and Z strings
print(op)                         # the Pauli strings and coefficients
print("constant offset:", offset)

# ---------------------------------------------------------------- 4. sanity check
# The Hamiltonian is diagonal, so the ground state is just the minimum of the
# diagonal. Exact enumeration is fine for ~13 qubits.
n = qubo.get_num_vars()
diag = np.real(op.to_matrix(sparse=True).diagonal()) + offset
idx = int(np.argmin(diag))

# Qiskit is little-endian: qubit k <-> bit k of the basis-state index
bits = np.array([(idx >> k) & 1 for k in range(n)])
x = conv.interpret(bits)          # back to original variables [x1,x2,x3,x4,t,b]
print("ground energy:", diag[idx])
print("decoded solution (x1,x2,x3,x4,t,b):", x)

# brute-force check of the formula itself
def f(x1, x2, x3, x4):
    return min(max(x1 + x3, x1 + x2 + x4), max(x3 + x1, x3 + x2 + x4))

x1, x2, x3, x4 = x[:4]
print("true f at decoded x:", f(x1, x2, x3, x4))   # should equal the ground energy
                                                  # (while f is your only objective)

# ---------------------------------------------------------------- 5. run it
# QAOA (small instances only):
#   from qiskit_algorithms import QAOA
#   from qiskit_algorithms.optimizers import COBYLA
#   from qiskit.primitives import Sampler
#   from qiskit_optimization.algorithms import MinimumEigenOptimizer
#   res = MinimumEigenOptimizer(QAOA(Sampler(), COBYLA(), reps=2)).solve(qubo)
#   print(conv.interpret(res.x))

binary variables / qubits: 13
SparsePauliOp(['IIIIIIIIIIZII', 'IIIIIIIIZIIII', 'IIIIIIIZIIIII', 'IIIIIIIIIIIZI', 'IIIIIIIIIZIII', 'IZIIIIIIIIIII', 'ZIIIIIIIIIIII', 'IIIIIIIIIIIIZ', 'IIIIIIIIZIIIZ', 'IIIIIIIZIIIIZ', 'IIIIIIZIIIIIZ', 'IIIIIZIIIIIIZ', 'IIIIIZIIIIIII', 'IIIIZIIIIIIIZ', 'IIIIZIIIIIIII', 'IIIIIIIIIZIZI', 'IIIIIIIIZIIZI', 'IIIIIIIZIIIZI', 'IZIIIIIIIIIZI', 'ZIIIIIIIIIIZI', 'IIIIIIIIZIZII', 'IIIIIIIZIIZII', 'IIIIIIZIIIZII', 'IIIZIIIIIIZII', 'IIIZIIIIIIIII', 'IIZIIIIIIIZII', 'IIZIIIIIIIIII', 'IIIIIIIIZZIII', 'IIIIIIIZIZIII', 'IZIIIIIIIZIII', 'ZIIIIIIIIZIII', 'IIIIIIIZZIIII', 'IIIIIZIIZIIII', 'IIIIZIIIZIIII', 'IIIZIIIIZIIII', 'IIZIIIIIZIIII', 'IZIIIIIIZIIII', 'ZIIIIIIIZIIII', 'IIIIIZIZIIIII', 'IIIIZIIZIIIII', 'IIIZIIIZIIIII', 'IIZIIIIZIIIII', 'IZIIIIIZIIIII', 'ZIIIIIIZIIIII', 'IIIIZZIIIIIII', 'IIZZIIIIIIIII', 'ZZIIIIIIIIIII'],
              coeffs=[ -2.75+0.j,  12.  +0.j,  24.  +0.j,  -7.5 +0.j,  -7.5 +0.j,  -7.5 +0.j,
  -7.5 +0.j,  -2.75+0.j,  -2.5 +0.j,  -5.  +0.j,   0.25+0.j, 